# AIMO3 Baseline Notebook — Day2 Skeleton

**目的**: Kaggle 上で提出成立する最小 baseline の確認  
**solver_type**: `placeholder` (答え = 0 固定)  
**Day2 スコープ**: 成立確認のみ。accuracy 議論・モデル比較は Day3 以降  

---

## BLOCKED 条件（実行前に確認すること）

| 条件 | チェック方法 | 対処 |
|---|---|---|
| dataset path 不明 | Cell 3 実行 | dataset を Add |
| model path 不明 | Cell 4 実行 | dataset を Add (model は Day3 以降) |
| `torch` import 失敗 | Cell 2 実行 | `pip install torch` |
| `pandas` import 失敗 | Cell 2 実行 | `pip install pandas` |
| submission.csv 生成失敗 | Cell 10 実行 | Cell 6〜9 のエラーを確認 |

> **注意**: BLOCKED 条件に該当する場合は先に進まず、該当セルのエラーを記録する。

## Cell 1: 設定・定数定義

In [ ]:
# ============================================================
# 設定・定数
# ============================================================
import os

# solver モード — 絶対に変更しないこと（Day2 は placeholder 固定）
SOLVER_TYPE = "placeholder"  # "placeholder" | "real"
assert SOLVER_TYPE == "placeholder", (
    f"Day2 は placeholder のみ許可。現在: {SOLVER_TYPE!r}"
)

# Kaggle データセットパス候補（存在確認は Cell 3 で実施）
DATASET_PATH_CANDIDATES = [
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3",
    "/kaggle/input/aimo-2025-ai-mathematical-olympiad-progress",
    "/kaggle/input/aimo3",
]

# モデルパス候補（存在確認は Cell 4 で実施）
MODEL_PATH_CANDIDATES = [
    "/kaggle/input/nvidia-openmath-nemotron-14b",
    "/kaggle/input/openmath-nemotron-14b-kaggle",
    "/kaggle/input/numina-math-7b-tir",
    "/kaggle/input/deepseek-math-7b-instruct",
    "/kaggle/input/qwen2-5-math-7b-instruct",
    "/kaggle/input/internlm2-math-plus-7b",
]

# 出力パス
OUTPUT_DIR = "/kaggle/working"
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")

print(f"SOLVER_TYPE : {SOLVER_TYPE}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"SUBMISSION  : {SUBMISSION_PATH}")

## Cell 2: 環境確認（Python / GPU / CUDA / torch）

In [ ]:
# ============================================================
# 環境確認
# BLOCKED 条件: import 失敗が 1件でもあれば BLOCKED
# ============================================================
import sys
import subprocess

print("=" * 50)
print("Python:", sys.version)
print("Platform:", sys.platform)

# pandas
try:
    import pandas as pd
    print(f"pandas    : {pd.__version__}  OK")
except ImportError as e:
    print(f"pandas    : MISSING — {e}")
    print("BLOCKED: pandas not available")

# torch
try:
    import torch
    print(f"torch     : {torch.__version__}  OK")
    cuda_available = torch.cuda.is_available()
    print(f"CUDA avail: {cuda_available}")
    if cuda_available:
        print(f"CUDA ver  : {torch.version.cuda}")
        print(f"GPU count : {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            name = torch.cuda.get_device_name(i)
            mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f"  GPU[{i}]   : {name}  ({mem:.1f} GB)")
    else:
        print("WARNING: CUDA not available — GPU inference impossible")
except ImportError as e:
    print(f"torch     : MISSING — {e}")
    print("BLOCKED: torch not available")

# transformers (モデル推論に必要。Day2 では import 確認のみ)
try:
    import transformers
    print(f"transformers: {transformers.__version__}  OK")
except ImportError as e:
    print(f"transformers: MISSING — {e}")
    print("WARNING: transformers not available (needed for Day3+ real solver)")

# nvidia-smi
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    if smi.returncode == 0:
        print("nvidia-smi:", smi.stdout.strip())
    else:
        print("nvidia-smi: not available")
except Exception as e:
    print(f"nvidia-smi: error — {e}")

print("=" * 50)

## Cell 3: Dataset path 確認

In [ ]:
# ============================================================
# Dataset path 確認
# BLOCKED 条件: DATASET_PATH が None のまま → BLOCKED
# ============================================================
import os
import glob

DATASET_PATH = None

print("=== Dataset path 候補 ====")
for candidate in DATASET_PATH_CANDIDATES:
    exists = os.path.isdir(candidate)
    print(f"  {'OK' if exists else '--'}  {candidate}")
    if exists and DATASET_PATH is None:
        DATASET_PATH = candidate

print()
if DATASET_PATH is None:
    print("BLOCKED: dataset path が見つかりません。")
    print("  → Kaggle で Add Data から公式 dataset を追加してください。")
    raise RuntimeError("BLOCKED: dataset path not found")
else:
    print(f"DATASET_PATH: {DATASET_PATH}")
    all_files = glob.glob(os.path.join(DATASET_PATH, "**", "*"), recursive=True)
    print("ファイル一覧:")
    for f in sorted(all_files):
        size = os.path.getsize(f) if os.path.isfile(f) else "-"
        print(f"  {f}  ({size} bytes)")

# 必須ファイルの確認
for required in ["test.csv", "sample_submission.csv"]:
    p = os.path.join(DATASET_PATH, required)
    if os.path.isfile(p):
        print(f"  FOUND: {required}")
    else:
        print(f"  MISSING: {required} — BLOCKED")
        raise FileNotFoundError(f"BLOCKED: {required} not found in {DATASET_PATH}")

## Cell 4: Model path 確認（Day2 では optional）

In [ ]:
# ============================================================
# Model path 確認
# Day2 では BLOCKED にはしない（placeholder は model 不要）
# Day3 以降の実 solver 用に候補を確認するのみ
# ============================================================

MODEL_PATH = None

print("=== Model path 候補 ===")
for candidate in MODEL_PATH_CANDIDATES:
    exists = os.path.isdir(candidate)
    print(f"  {'OK' if exists else '--'}  {candidate}")
    if exists and MODEL_PATH is None:
        MODEL_PATH = candidate
        files = os.listdir(candidate)[:5]
        print(f"       → 先頭5ファイル: {files}")

print()
if MODEL_PATH is None:
    print("WARNING: model path が見つかりません。")
    print("  → Day2 (placeholder solver) では不要です。")
    print("  → Day3 以降は Add Data でモデルを追加してください。")
    print("  → 推奨: nvidia/OpenMath-Nemotron-14B-Kaggle (AIMO2 優勝)")
else:
    print(f"MODEL_PATH: {MODEL_PATH}")

print("Day2 は placeholder solver のため model は不要 — PASS")

## Cell 5: Output path 確認

In [ ]:
# ============================================================
# Output path 確認
# BLOCKED 条件: /kaggle/working が書き込み不可 → BLOCKED
# ============================================================

print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"  exists   : {os.path.isdir(OUTPUT_DIR)}")
print(f"  writable : {os.access(OUTPUT_DIR, os.W_OK)}")

if not os.path.isdir(OUTPUT_DIR):
    print("BLOCKED: OUTPUT_DIR が存在しません")
    raise RuntimeError(f"BLOCKED: {OUTPUT_DIR} does not exist")

if not os.access(OUTPUT_DIR, os.W_OK):
    print("BLOCKED: OUTPUT_DIR が書き込み不可です")
    raise RuntimeError(f"BLOCKED: {OUTPUT_DIR} is not writable")

# ログ保存先の確認
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
print(f"LOG_DIR   : {LOG_DIR}  (作成済み)")
print("Output path PASS")

## Cell 6: AIMO3 データ読み込み

In [ ]:
# ============================================================
# AIMO3 公式データ読み込み
# BLOCKED 条件: DATASET_PATH が None → Cell 3 で止まっているはず
# ============================================================
import pandas as pd

test_path   = os.path.join(DATASET_PATH, "test.csv")
sample_path = os.path.join(DATASET_PATH, "sample_submission.csv")

test_df   = pd.read_csv(test_path)
sample_df = pd.read_csv(sample_path)

print("=== test.csv ===")
print(f"  shape  : {test_df.shape}")
print(f"  columns: {list(test_df.columns)}")
print(f"  dtypes : {test_df.dtypes.to_dict()}")
print()
print("先頭 3 行:")
print(test_df.head(3).to_string())

print()
print("=== sample_submission.csv ===")
print(f"  shape  : {sample_df.shape}")
print(f"  columns: {list(sample_df.columns)}")
print()
print(sample_df.head(3).to_string())

# 問題数・ID 確認
NUM_PROBLEMS = len(test_df)
print(f"\n問題数 (NUM_PROBLEMS): {NUM_PROBLEMS}")

# 問題テキストのカラム名を自動検出
PROBLEM_COL = None
ANSWER_COL  = "answer"
ID_COL      = "id"

for candidate in ["problem", "problem_text", "question", "text"]:
    if candidate in test_df.columns:
        PROBLEM_COL = candidate
        break

if PROBLEM_COL is None:
    print(f"BLOCKED: problem カラムが見つかりません。columns={list(test_df.columns)}")
    raise RuntimeError("BLOCKED: problem column not found")

if ID_COL not in test_df.columns:
    print(f"BLOCKED: id カラムが見つかりません。columns={list(test_df.columns)}")
    raise RuntimeError("BLOCKED: id column not found")

print(f"ID_COL      : {ID_COL!r}")
print(f"PROBLEM_COL : {PROBLEM_COL!r}")

## Cell 7: 1問取り出し・問題文確認

In [ ]:
# ============================================================
# 1問取り出し・問題文確認
# ============================================================

FIRST_ROW   = test_df.iloc[0]
FIRST_ID    = FIRST_ROW[ID_COL]
FIRST_PROB  = FIRST_ROW[PROBLEM_COL]

print("=== 問題 #0 ===")
print(f"  id     : {FIRST_ID!r}")
print(f"  problem: {FIRST_PROB[:300]}{'...' if len(str(FIRST_PROB)) > 300 else ''}")
print(f"  (全長 {len(str(FIRST_PROB))} 文字)")

## Cell 8: Placeholder solver 定義

> **注意**: `SOLVER_TYPE = "placeholder"` の確認が Cell 1 で行われているため、  
> このセルは Day2 専用。real solver への差し替えは Day3 以降に行う。

In [ ]:
# ============================================================
# Placeholder solver
# solver_type = "placeholder" 固定
# 常に 0 を返す。絶対に real solver と混同しない。
# ============================================================

def solve(problem_id: str, problem_text: str) -> int:
    """Placeholder solver — always returns 0.

    Args:
        problem_id  : problem ID string (e.g., "2025-0001")
        problem_text: LaTeX-formatted problem text

    Returns:
        int: predicted answer (0 for placeholder)
    """
    assert SOLVER_TYPE == "placeholder", (
        f"solver_type mismatch: expected 'placeholder', got {SOLVER_TYPE!r}"
    )
    return 0


# 動作確認
result = solve(FIRST_ID, FIRST_PROB)
print(f"solve({FIRST_ID!r}, ...) = {result!r}")
assert isinstance(result, int), f"answer must be int, got {type(result)}"
assert 0 <= result <= 999, f"AIME answer out of range: {result}"
print(f"solver_type : {SOLVER_TYPE}")
print("Placeholder solver PASS")

## Cell 9: 1問タイマー計測

In [ ]:
# ============================================================
# 1問タイマー計測
# placeholder は即時なので elapsed ≈ 0s
# real solver に差し替えたときの参考にする
# ============================================================
import time

start = time.monotonic()
answer_0 = solve(FIRST_ID, FIRST_PROB)
elapsed = time.monotonic() - start

print(f"=== 1問タイマー計測 ===")
print(f"  problem_id  : {FIRST_ID!r}")
print(f"  answer      : {answer_0}")
print(f"  elapsed_sec : {elapsed:.4f}s")
print()
print(f"  参考換算:")
print(f"    {NUM_PROBLEMS} 問換算 : {elapsed * NUM_PROBLEMS / 60:.2f} min  ({elapsed * NUM_PROBLEMS:.1f} sec)")
print(f"    50 問換算           : {elapsed * 50 / 60:.2f} min  ({elapsed * 50:.1f} sec)")
print(f"    100 問換算          : {elapsed * 100 / 60:.2f} min  ({elapsed * 100:.1f} sec)")
print()
print(f"  Kaggle runtime 上限 (GPU): 9h = {9*3600}s")
print(f"  1問あたりの許容時間      : {9*3600 / max(NUM_PROBLEMS, 1):.1f}s")
if elapsed > 0:
    margin = 9*3600 / max(NUM_PROBLEMS, 1) / elapsed
    print(f"  余裕倍率 (現在の solver) : {margin:.0f}x")

## Cell 10: 全問 placeholder 実行 → submission.csv 生成

In [ ]:
# ============================================================
# 全問実行 (placeholder)
# BLOCKED 条件: submission.csv が生成できない → BLOCKED
# ============================================================
import time

print(f"=== 全問実行 (solver_type={SOLVER_TYPE!r}) ===")
print(f"  問題数: {NUM_PROBLEMS}")
print()

results = []
batch_start = time.monotonic()

for _, row in test_df.iterrows():
    prob_id   = row[ID_COL]
    prob_text = row[PROBLEM_COL]

    t0     = time.monotonic()
    answer = solve(prob_id, prob_text)
    t1     = time.monotonic()

    results.append({
        "id"          : prob_id,
        "answer"      : answer,
        "elapsed_sec" : round(t1 - t0, 6),
        "solver_type" : SOLVER_TYPE,
    })

batch_elapsed = time.monotonic() - batch_start

result_df = pd.DataFrame(results)

print(f"  total           : {len(result_df)}")
print(f"  batch_elapsed   : {batch_elapsed:.2f}s")
print(f"  avg_per_problem : {batch_elapsed / len(result_df):.4f}s")
print(f"  max_per_problem : {result_df['elapsed_sec'].max():.4f}s")
print(f"  solver_type     : {SOLVER_TYPE}")
print()
print(result_df[[ID_COL, "answer", "elapsed_sec"]].to_string())

## Cell 11: submission.csv 保存・検証

In [ ]:
# ============================================================
# submission.csv 保存・検証
# BLOCKED 条件: 保存失敗 or フォーマット不一致 → BLOCKED
# ============================================================

# sample_submission の形式に合わせる
submission_df = result_df[[ID_COL, "answer"]].copy()
submission_df.columns = ["id", "answer"]

# 保存
submission_df.to_csv(SUBMISSION_PATH, index=False)

# 検証
assert os.path.isfile(SUBMISSION_PATH), f"BLOCKED: {SUBMISSION_PATH} が存在しません"
verify_df = pd.read_csv(SUBMISSION_PATH)

print(f"=== submission.csv 検証 ===")
print(f"  path    : {SUBMISSION_PATH}")
print(f"  size    : {os.path.getsize(SUBMISSION_PATH)} bytes")
print(f"  shape   : {verify_df.shape}")
print(f"  columns : {list(verify_df.columns)}")
print()
print(verify_df.head(5).to_string())
print()

# フォーマット確認
assert list(verify_df.columns) == ["id", "answer"], (
    f"BLOCKED: columns mismatch. expected ['id', 'answer'], got {list(verify_df.columns)}"
)
assert len(verify_df) == NUM_PROBLEMS, (
    f"BLOCKED: row count mismatch. expected {NUM_PROBLEMS}, got {len(verify_df)}"
)

# sample_submission との ID 一致確認
if set(verify_df["id"]) == set(sample_df["id"]):
    print("ID 一致: OK")
else:
    missing_ids = set(sample_df["id"]) - set(verify_df["id"])
    extra_ids   = set(verify_df["id"]) - set(sample_df["id"])
    print(f"BLOCKED: ID 不一致。missing={missing_ids}, extra={extra_ids}")
    raise RuntimeError("BLOCKED: submission ID mismatch")

print(f"submission.csv 生成 PASS")
print(f"solver_type = {SOLVER_TYPE!r} — placeholder であることを確認")

## Cell 12: BLOCKED 条件まとめ（実行結果の記録用）

In [ ]:
# ============================================================
# BLOCKED 条件まとめ
# このセルが正常終了 = Day2 PASS
# ============================================================

checks = {
    "python_version"           : sys.version,
    "cuda_available"           : torch.cuda.is_available() if 'torch' in dir() else "torch not imported",
    "dataset_path"             : DATASET_PATH if DATASET_PATH else "NOT FOUND — BLOCKED",
    "model_path"               : MODEL_PATH if MODEL_PATH else "NOT FOUND (OK for Day2)",
    "output_dir_writable"      : os.access(OUTPUT_DIR, os.W_OK),
    "submission_csv_exists"    : os.path.isfile(SUBMISSION_PATH),
    "submission_rows"          : len(pd.read_csv(SUBMISSION_PATH)) if os.path.isfile(SUBMISSION_PATH) else "N/A",
    "solver_type"              : SOLVER_TYPE,
}

print("=== Day2 完了チェック ===")
all_pass = True
for key, val in checks.items():
    status = "OK" if "BLOCKED" not in str(val) else "BLOCKED"
    if status == "BLOCKED":
        all_pass = False
    print(f"  [{status}]  {key}: {val}")

print()
if all_pass:
    print("Day2: PASS — submission.csv 生成成立")
else:
    print("Day2: BLOCKED — 上記の BLOCKED 項目を解決してください")
    raise RuntimeError("Day2 BLOCKED")

---

## 注意事項

| 項目 | 説明 |
|---|---|
| `solver_type` | `"placeholder"` のみ許可（Day2）。`"real"` への変更は Day3 以降 |
| `submission.csv` | `answer` は常に `0`（placeholder）— public score は 0/50 |
| AIME answer 範囲 | 整数 0〜999。Kaggle では plain int で submit |
| runtime 確認 | Cell 9 の換算値でタイムバジェットを確認すること |
| model | Day3 以降: `nvidia/OpenMath-Nemotron-14B-Kaggle` (AIMO2 優勝) を推奨 |

## Day3 以降の手順

1. Cell 1 の `SOLVER_TYPE` を `"real"` に変更
2. Cell 4 でモデルパスを確認
3. Cell 8 の `solve()` を実 solver に差し替え
4. Cell 9 でタイマー計測（32問換算 < runtime 制限を確認）
5. 全件実行 → submission.csv 生成 → Kaggle submit